# Vision 실습 문제 (Practice)

각 셀에서 `### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!` 부분을 채워보세요.

## 문제 1. CIFAR-10 Normalize (정규화)

In [ ]:
import torchvision.transforms as T

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)

train_tfms = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
])

## 문제 2. Top-1 정확도(Accuracy) 계산

In [ ]:
def accuracy_top1(logits, targets):
    preds = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
    return (preds == targets).float().mean().item()

## 문제 3. 학습 루프의 핵심 3단계 (optimizer.zero_grad / backward / step)

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
        logits = model(images)
        loss = criterion(logits, labels)
        ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
        ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

## 문제 4. SGD + Momentum Optimizer 생성

In [ ]:
import torch
import torch.nn as nn

criterion = nn.CrossEntropyLoss()
optimizer = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

## 문제 5. ViT 패치 임베딩 (einops Rearrange 패턴)

In [ ]:
from einops.layers.torch import Rearrange
import torch.nn as nn

patch_height = 4
patch_width = 4
patch_dim = 3 * patch_height * patch_width
dim = 64

to_patch_embedding = nn.Sequential(
    Rearrange(### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!, p1=patch_height, p2=patch_width),
    nn.Linear(patch_dim, dim),
)

## 문제 6. Scaled Dot-Product Attention (Q·Kᵀ 유사도와 softmax)

In [ ]:
import torch
from torch import nn, einsum
from einops import rearrange

class Attention(nn.Module):
    def __init__(self, dim, heads=4, dim_head=64):
        super().__init__()
        inner_dim = dim_head * heads
        self.heads = heads
        self.scale = dim_head ** -0.5
        self.attend = nn.Softmax(dim=-1)
        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)
        self.to_out = nn.Linear(inner_dim, dim)

    def forward(self, x):
        b, n, _, h = *x.shape, self.heads
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=h), qkv)

        dots = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
        attn = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

        out = einsum('b h i j, b h j d -> b h i d', attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        return self.to_out(out)

## 문제 7. ViT CLS 토큰 결합 & 위치 임베딩 더하기

In [ ]:
import torch
from torch import nn
from einops import repeat

class ViTForward(nn.Module):
    def __init__(self, num_patches, dim):
        super().__init__()
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, dim))

    def forward(self, x):
        b, n, _ = x.shape
        cls_tokens = repeat(self.cls_token, '1 1 d -> b 1 d', b=b)
        x = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
        x = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
        return x

## 문제 8. ViT 분류 표현 추출: CLS 토큰 선택 vs mean pooling

In [ ]:
def select_feature(x, pool):
    x = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
    return x

## 문제 9. DETR 입력 전처리 파이프라인

In [ ]:
import torchvision.transforms as T

transform = T.Compose([
    ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
    ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

## 문제 10. 박스 좌표 변환: (cx,cy,w,h) → (xmin,ymin,xmax,ymax)

In [ ]:
import torch

def box_cxcywh_to_xyxy(x):
    x_c, y_c, w, h = x.unbind(1)
    b = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
    return torch.stack(b, dim=1)

## 문제 11. DETR 예측 신뢰도 필터링

In [ ]:
probas = outputs['pred_logits'].softmax(-1)[0, :, :-1]
keep = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

## 문제 12. Forward Hook으로 중간 텐서 캡처

In [ ]:
conv_features, enc_attn_weights, dec_attn_weights = [], [], []

hooks = [
    model.backbone[-2].### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(
        lambda self, input, output: conv_features.append(output)
    ),
    model.transformer.encoder.layers[-1].self_attn.register_forward_hook(
        lambda self, input, output: enc_attn_weights.append(output[1])
    ),
    model.transformer.decoder.layers[-1].multihead_attn.register_forward_hook(
        lambda self, input, output: dec_attn_weights.append(output[1])
    ),
]

outputs = model(img)

for hook in hooks:
    ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

## 문제 13. U-Net DoubleConv 블록 구현

In [ ]:
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if mid_channels is None:
            mid_channels = out_channels
        self.net = nn.Sequential(
            ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
            ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
            ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
            ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
        )

    def forward(self, x):
        return self.net(x)

## 문제 14. U-Net Skip Connection 결합

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Up(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        diff_y = x2.size(2) - x1.size(2)
        diff_x = x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [diff_x // 2, diff_x - diff_x // 2,
                        diff_y // 2, diff_y - diff_y // 2])
        x = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
        return self.conv(x)

## 문제 15. Binary Segmentation 예측 마스크 생성

In [ ]:
import torch

@torch.no_grad()
def predict_mask(model, imgs):
    model.eval()
    logits = model(imgs)
    probs = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
    preds = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
    return preds

## 문제 16. DDPM Sinusoidal Timestep Embedding

In [ ]:
import math
import torch
import torch.nn as nn

class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        freq = math.log(10000) / (half_dim - 1)
        freq = torch.exp(torch.arange(half_dim, device=device) * -freq)
        args = time[:, None].float() * freq[None, :]
        emb = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
        return emb

## 문제 17. DDPM Timestep 주입(FiLM / AdaGN scale-shift)

In [ ]:
import torch.nn as nn

class Block(nn.Module):
    def __init__(self, dim, dim_out, groups=8):
        super().__init__()
        self.proj = nn.Conv2d(dim, dim_out, 3, padding=1)
        self.norm = nn.GroupNorm(groups, dim_out)
        self.act = nn.SiLU()

    def forward(self, x, scale_shift=None):
        x = self.proj(x)
        x = self.norm(x)
        if scale_shift is not None:
            scale, shift = scale_shift
            ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
        x = self.act(x)
        return x

## 문제 18. DDPM U-Net의 Self-Attention(einsum)

In [ ]:
import torch
import torch.nn as nn
from einops import rearrange

class DDPMAttention(nn.Module):
    def __init__(self, dim, heads=4, dim_head=32):
        super().__init__()
        self.scale = dim_head ** -0.5
        self.heads = heads
        hidden_dim = dim_head * heads
        self.to_qkv = nn.Conv2d(dim, hidden_dim * 3, 1, bias=False)
        self.to_out = nn.Conv2d(hidden_dim, dim, 1)

    def forward(self, x):
        b, c, h, w = x.shape
        qkv = self.to_qkv(x).chunk(3, dim=1)
        q, k, v = map(lambda t: rearrange(t, 'b (h c) x y -> b h c (x y)', h=self.heads), qkv)
        q = q * self.scale
        sim = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
        sim = sim - sim.amax(dim=-1, keepdim=True).detach()
        attn = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
        out = torch.einsum('b h i j, b h d j -> b h i d', attn, v)
        out = rearrange(out, 'b h (x y) d -> b (h d) x y', x=h, y=w)
        return self.to_out(out)

## 문제 19. Stable Diffusion 파이프라인으로 이미지 생성

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

pipe = StableDiffusionPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4",
    torch_dtype=torch.float16
).to("cuda")

prompt = "A futuristic cityscape at sunset with flying cars"
image = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

## 문제 20. 재현 가능한 SD 생성: generator seed + negative_prompt

In [ ]:
import torch

prompt = "A beautiful watercolor painting of a green forest"
negative = "woman, red flowers, red color"
generator = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

image = pipe(
    prompt,
    negative_prompt=negative,
    generator=generator,
    num_inference_steps=30,
    guidance_scale=7.5,
).images[0]